# 🚀 Apache Spark: Arhitectură și Execuție

Acest document sintetizează conceptele fundamentale despre modul în care **Apache Spark** gestionează resursele și execută sarcinile, evidențiind relația cu ecosistemul Hadoop și mecanismele de procesare distribuită.

---

## 🏗️ Arhitectura Clusterului: Brains & Brawn

Într-un mediu Spark, responsabilitățile sunt clar împărțite între două entități principale:

### 1. Driver (The Brain)
*   **Rol:** Inima aplicației Spark.
*   **Responsabilități:**
    *   Menține informațiile despre starea aplicației.
    *   Analizează, distribuie și planifică sarcinile (**Tasks**) pe executori.
    *   Găzduiește `SparkSession`.
    *   Colectează rezultatele finale pentru a le afișa utilizatorului.

### 2. Executor (The Brawn)
*   **Rol:** Unitățile de execuție care fac "munca grea".
*   **Responsabilități:**
    *   Execută codul primit de la Driver.
    *   Raportează Driver-ului starea execuției (succes/eroare).
    *   Stochează datele în memorie (cache) sau pe disc pentru procesări ulterioare.

---

## 📊 Ierarhia de Execuție

Spark sparge codul tău în unități de lucru din ce în ce mai mici pentru a le putea distribui:

1.  **Application:** Programul tău complet (ex: un script Python sau un job JAR).
2.  **Job:** Lansat automat de fiecare dată când apelezi o **Action** (ex: `.count()`, `.save()`, `.collect()`).
3.  **Stage:** O subdiviziune a unui Job. Un job este împărțit în Stage-uri ori de câte ori apare un **Shuffle**.
4.  **Task:** Cea mai mică unitate. Un task este trimis de Driver către un Executor pentru a procesa o singură partiție de date.

---

## 🔄 Mecanismul de Shuffle & Stages

**Shuffle-ul** este punctul critic în performanța Spark. Acesta reprezintă procesul de redistribuire a datelor între executori astfel încât datele cu aceleași chei să ajungă în același loc.

### Exemplu: `count()`
Pentru a număra elementele dintr-un dataset distribuit, Spark trece prin două etape:

*   **Stage 1: Local Count**
    *   Fiecare executor își numără rândurile din partițiile locale.
    *   Rezultatul este un număr parțial.
*   **Stage 2: Global Count (The Shuffle/Final Aggregation)**
    *   Se face un shuffle al rezultatelor parțiale.
    *   Acestea sunt adunate (de obicei la nivel de Driver sau un executor central) pentru a obține totalul.

> [!IMPORTANT]
> **Persistența în Shuffle:** În timpul unui shuffle, datele sunt scrise pe disc (**persisted/shuffle files**). Acest lucru asigură **Fault Tolerance**: dacă un nod moare, nu trebuie să reluăm tot job-ul, ci doar de la ultimul stage salvat.

---

## 🛠️ Tipuri de Transformări

| Tip | Descriere | Shuffle? | Exemplu |
| :--- | :--- | :--- | :--- |
| **Narrow** | Datele necesare sunt într-o singură partiție. | **NU** | `filter()`, `map()` |
| **Wide** | Datele necesare sunt împrăștiate în mai multe partiții. | **DA** | `groupBy()`, `join()` |

---

## 🌐 Stack-ul Modern de Big Data

Deși Spark poate rula pe **Hadoop (HDFS + YARN)**, tendința actuală în companiile mari este migrarea către:
*   **Stocare:** Object Storage (AWS S3, Azure Data Lake).
*   **Orchestrare:** **Kubernetes (K8s)** în loc de YARN pentru o scalabilitate mai elastică.
*   **Format:** Delta Lake / Iceberg (pentru a aduce funcționalități de bază de date peste fișiere).

---

> *"Spark nu înlocuiește Hadoop, ci îi dă viteză de Formula 1."*

---

E ceva ce ai vrea să completezi sau să detaliem mai mult în această structură?

### Imports

In [1]:
%pip list | grep plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install plotly

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 14.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.4/449.4 KB 40.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_unixtime,
    to_timestamp,
    min,
    max,
    sum,
    avg,
    col,
    countDistinct,
    broadcast,
    date_trunc,
    count
)
from pyspark.sql import Window
import pyspark.sql.functions as F
import plotly.express as px

### Session

In [7]:
spark = (
    SparkSession.builder.appName("IOT")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .master("spark://master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.version

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 11:34:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.0'

In [8]:
filepaths = ["/home/ubuntu/data/IOT/IOT/CTU-IoT-Malware-Capture-1-1conn.log.labeled.csv",
             "/home/ubuntu/data/IOT/IOT/CTU-IoT-Malware-Capture-3-1conn.log.labeled.csv"]

df = spark.read.option("delimiter", "|").csv(filepaths, inferSchema=True, header=True)
df.show(n=5)

+-------------------+------------------+---------------+---------+---------------+---------+-----+-------+--------+----------+----------+----------+----------+----------+------------+-------+---------+-------------+---------+-------------+--------------+---------+--------------------+
|                 ts|               uid|      id.orig_h|id.orig_p|      id.resp_h|id.resp_p|proto|service|duration|orig_bytes|resp_bytes|conn_state|local_orig|local_resp|missed_bytes|history|orig_pkts|orig_ip_bytes|resp_pkts|resp_ip_bytes|tunnel_parents|    label|      detailed-label|
+-------------------+------------------+---------------+---------+---------------+---------+-----+-------+--------+----------+----------+----------+----------+----------+------------+-------+---------+-------------+---------+-------------+--------------+---------+--------------------+
|1.525879831015811E9|CUmrqr4svHuSXJy5z7|192.168.100.103|  51524.0| 65.127.233.163|     23.0|  tcp|      -|2.999051|         0|         0|     

In [10]:
df.printSchema()

root
 |-- ts: double (nullable = true)
 |-- uid: string (nullable = true)
 |-- id.orig_h: string (nullable = true)
 |-- id.orig_p: double (nullable = true)
 |-- id.resp_h: string (nullable = true)
 |-- id.resp_p: double (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- orig_bytes: string (nullable = true)
 |-- resp_bytes: string (nullable = true)
 |-- conn_state: string (nullable = true)
 |-- local_orig: string (nullable = true)
 |-- local_resp: string (nullable = true)
 |-- missed_bytes: double (nullable = true)
 |-- history: string (nullable = true)
 |-- orig_pkts: double (nullable = true)
 |-- orig_ip_bytes: double (nullable = true)
 |-- resp_pkts: double (nullable = true)
 |-- resp_ip_bytes: double (nullable = true)
 |-- tunnel_parents: string (nullable = true)
 |-- label: string (nullable = true)
 |-- detailed-label: string (nullable = true)



### Pre-processing

In [12]:
df = df.withColumn("dt", from_unixtime("ts")).withColumn("dt", to_timestamp("dt"))

In [13]:
df.show(n=3)

[Stage 4:>                                                          (0 + 1) / 1]

+-------------------+------------------+---------------+---------+--------------+---------+-----+-------+--------+----------+----------+----------+----------+----------+------------+-------+---------+-------------+---------+-------------+--------------+---------+--------------------+-------------------+
|                 ts|               uid|      id.orig_h|id.orig_p|     id.resp_h|id.resp_p|proto|service|duration|orig_bytes|resp_bytes|conn_state|local_orig|local_resp|missed_bytes|history|orig_pkts|orig_ip_bytes|resp_pkts|resp_ip_bytes|tunnel_parents|    label|      detailed-label|                 dt|
+-------------------+------------------+---------------+---------+--------------+---------+-----+-------+--------+----------+----------+----------+----------+----------+------------+-------+---------+-------------+---------+-------------+--------------+---------+--------------------+-------------------+
|1.525879831015811E9|CUmrqr4svHuSXJy5z7|192.168.100.103|  51524.0|65.127.233.163|    

In [14]:
df = df.withColumnsRenamed(
    {
        "id.orig_h": "source_ip",
        "id.orig_p": "source_port",
        "id.resp_h": "dest_ip",
        "id.resp_p": "dest_port"
    }
)

In [15]:
df.select("ts").show(n=3)

+-------------------+
|                 ts|
+-------------------+
|1.525879831015811E9|
|1.525879831025055E9|
|1.525879831045045E9|
+-------------------+
only showing top 3 rows



In [16]:
df.agg(
    min("dt").alias("min_date"),
    max("dt").alias("max_date")
).show()

[Stage 6:============================================>              (3 + 1) / 4]

+-------------------+-------------------+
|           min_date|           max_date|
+-------------------+-------------------+
|2018-05-09 15:30:31|2018-05-21 07:04:46|
+-------------------+-------------------+



### Shape

In [18]:
df.count(), len(df.columns)

(1164851, 24)

In [22]:
to_analyse = [
    "source_ip",
    "source_port",
    "dest_ip",
    "dest_port",
    "proto",
    "service",
    "duration",
    "label",
    "detailed-label"
]

unique_counts = df.agg(*(countDistinct(col(c)).alias(c) for c in to_analyse))
print(unique_counts)

unique_counts = unique_counts.first()
static_cols = [c for c in unique_counts.asDict() if unique_counts[c] == 1]
print("Dataset has ", len(static_cols), " static columns: ", static_cols)
df = df.drop(*static_cols)

DataFrame[source_ip: bigint, source_port: bigint, dest_ip: bigint, dest_port: bigint, proto: bigint, service: bigint, duration: bigint, label: bigint, detailed-label: bigint]


[Stage 23:==============>                                           (1 + 3) / 4]

Dataset has  0  static columns:  []


### Count Nulls

In [24]:
df = df.replace("-", None)
remaining_cols = [f for f in to_analyse if f not in static_cols]
df.select(
    [count(F.when(F.isnan(c) | col(c).isNull(), c)).alias(c) for c in remaining_cols]
).show()

[Stage 27:===========================================>              (3 + 1) / 4]

+---------+-----------+-------+---------+-----+-------+--------+-----+--------------+
|source_ip|source_port|dest_ip|dest_port|proto|service|duration|label|detailed-label|
+---------+-----------+-------+---------+-----+-------+--------+-----+--------------+
|        0|          0|      0|        0|    0|1155702|  870244|    0|        473811|
+---------+-----------+-------+---------+-----+-------+--------+-----+--------------+

